# Geração e preparação dos dados

Gera dados sintéticos para o case: 100.000 pedidos, 200.000 itens e 5.000 produtos. As descrições são artificiais e não reproduzem registros do dataset Amazon. Os CSVs são gravados em `data/bronze/` na raiz do repositório.

Instale as dependências antes de executar: `pip install -r ../src/requirements.txt`.

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
import random
import uuid

import pandas as pd
from faker import Faker

SEED = 20260929
N_ORDERS = 100_000
ITEMS_PER_ORDER = 2
N_PRODUCTS = 5_000
REPOSITORY_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = REPOSITORY_ROOT / "data" / "bronze"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
Faker.seed(SEED)
fake = Faker("en_US")
categories = ["home", "electronics", "kitchen", "sports", "beauty", "books", "fashion"]
statuses = ["delivered", "shipped", "processing", "canceled"]
status_weights = [0.82, 0.08, 0.07, 0.03]

In [ ]:
product_ids = [str(uuid.uuid4()) for _ in range(N_PRODUCTS)]
products = []
for product_id in product_ids:
    category = random.choice(categories)
    products.append({
        "product_id": product_id,
        "product_category_name_raw": f"  {category.title()}  ",
        "title_raw": f"{fake.catch_phrase()} {fake.word().title()}",
        "description_raw": fake.paragraph(nb_sentences=8),
    })
products_df = pd.DataFrame(products)

orders = []
order_items = []
start_date = datetime(2025, 1, 1)
end_date = datetime(2026, 8, 31)
for _ in range(N_ORDERS):
    order_id = str(uuid.uuid4())
    customer_id = str(uuid.uuid4())
    purchase_time = fake.date_time_between(start_date=start_date, end_date=end_date)
    status = random.choices(statuses, weights=status_weights, k=1)[0]
    delivered_time = (
        purchase_time + timedelta(days=random.randint(1, 20))
        if status == "delivered"
        else None
    )
    orders.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "order_status": status,
        "order_purchase_timestamp": purchase_time,
        "order_delivered_customer_date": delivered_time,
    })
    for item_id in range(1, ITEMS_PER_ORDER + 1):
        order_items.append({
            "order_id": order_id,
            "order_item_id": item_id,
            "product_id": random.choice(product_ids),
            "price": round(random.uniform(5, 1500), 2),
            "freight_value": round(random.uniform(0, 120), 2),
        })

orders_df = pd.DataFrame(orders)
order_items_df = pd.DataFrame(order_items)


In [ ]:
datasets = {
    "tb_orders.csv": orders_df,
    "tb_order_items.csv": order_items_df,
    "tb_products_unstructured.csv": products_df,
}
for filename, dataframe in datasets.items():
    dataframe.to_csv(OUTPUT_DIR / filename, index=False, encoding="utf-8")
    print(f"{filename}: {len(dataframe):,} registros")

assert len(orders_df) == N_ORDERS
assert len(order_items_df) == N_ORDERS * ITEMS_PER_ORDER
assert len(products_df) == N_PRODUCTS
assert sum(map(len, datasets.values())) > 100_000
assert orders_df["order_id"].is_unique
assert orders_df.columns.tolist() == [
    "order_id", "customer_id", "order_status",
    "order_purchase_timestamp", "order_delivered_customer_date",
]
assert order_items_df.columns.tolist() == [
    "order_id", "order_item_id", "product_id", "price", "freight_value",
]
assert products_df.columns.tolist() == [
    "product_id", "product_category_name_raw", "title_raw", "description_raw",
]
for filename, dataframe in datasets.items():
    sample = pd.read_csv(OUTPUT_DIR / filename, encoding="utf-8", nrows=3)
    assert sample.columns.tolist() == dataframe.columns.tolist()
    assert len(sample) == 3

print(f"Total: {sum(map(len, datasets.values())):,} registros; UTF-8 e colunas validados.")